In [11]:
import requests
import pandas as pd
import json


def fetch_financial_statements(stock_code, report_type, fiscal_dates=None, model_types="1", size=2000):
    """Fetches financial statements by stock code and report type (GET)."""
    report_key = str(report_type).upper().strip()
    report_aliases = {
        "Q": "QUARTER",
        "QUARTER": "QUARTER",
        "QUARTERLY": "QUARTER",
        "A": "ANNUAL",
        "Y": "ANNUAL",
        "YEAR": "ANNUAL",
        "YEARLY": "ANNUAL",
        "ANNUAL": "ANNUAL",
    }
    report_type = report_aliases.get(report_key)
    if report_type is None:
        raise ValueError("report_type must be one of: QUARTER, QUARTERLY, Q, ANNUAL, YEARLY, Y")

    url = "https://api-finfo.vndirect.com.vn/v4/financial_statements"
    query_parts = [
        f"code:{stock_code}",
        f"reportType:{report_type}",
        f"modelType:{model_types}",
    ]
    if fiscal_dates:
        query_parts.append(f"fiscalDate:{','.join(fiscal_dates)}")

    params = {
        "q": "~".join(query_parts),
        "sort": "fiscalDate:desc",
        "size": size,
    }
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "application/json",
    }

    try:
        response = requests.get(url, params=params, headers=headers, timeout=30)
        response.raise_for_status()
        data = response.json().get("data", [])
        if not data:
            print(f"No data found for {stock_code} ({report_type}).")
            return pd.DataFrame()

        df = pd.DataFrame(data)
        if "fiscalDate" in df.columns:
            df["fiscalDate"] = pd.to_datetime(df["fiscalDate"])
        return df
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data for {stock_code} ({report_type}): {e}")
        return pd.DataFrame()


def fetch_stock_price(stock_code, start_date, end_date):
    """Fetches stock prices by code and date range (GET)."""
    url = f"https://api-finfo.vndirect.com.vn/v4/stock_prices?sort=date&q=code:{stock_code}~date:gte:{start_date}~date:lte:{end_date}&size=15&page=1"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "application/json",
    }
    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        data = response.json().get("data", [])
        if not data:
            print(f"No stock price data found for {stock_code}.")
            return pd.DataFrame()
        df = pd.DataFrame(data)
        if "date" in df.columns:
            df["date"] = pd.to_datetime(df["date"])
        return df
    except requests.exceptions.RequestException as e:
        print(f"Error fetching stock prices for {stock_code}: {e}")
        return pd.DataFrame()


def fetch_financial_models(stock_code, model_types="1", notes=None, display_levels="0,1,2,3", size=999):
    """Fetches financial model metadata by stock code (GET)."""
    if notes is None:
        notes = "TT199/2014/TT-BTC,TT334/2016/TT-BTC,TT49/2014/TT-NHNN,TT202/2014/TT-BTC"

    url = "https://api-finfo.vndirect.com.vn/v4/financial_models"
    params = {
        "sort": "displayOrder:asc",
        "q": (
            f"codeList:{stock_code}"
            f"~modelType:{model_types}"
            f"~note:{notes}"
            f"~displayLevel:{display_levels}"
        ),
        "size": size,
    }
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "application/json",
    }

    try:
        response = requests.get(url, params=params, headers=headers, timeout=30)
        response.raise_for_status()
        data = response.json().get("data", [])
        if not data:
            print(f"No model data found for {stock_code}.")
            return pd.DataFrame()
        return pd.DataFrame(data)
    except requests.exceptions.RequestException as e:
        print(f"Error fetching financial models for {stock_code}: {e}")
        return pd.DataFrame()


def get_financial_ratios(stock_code, report_date):
    """Fetches financial ratios by stock code and report date (GET)."""
    ratio_codes = [
        "NET_SALES_TR_GRYOY", "NET_PROFIT_TR_GRYOY", "OPERATING_EBIT_TR_GRYOY",
        "GROSS_MARGIN_TR", "ROAA_TR_AVG5Q", "ROAE_TR_AVG5Q", "DEBT_TO_EQUITY_AQ",
        "DIVIDEND_YIELD", "CFO_TO_SALES_TR", "INTEREST_COVERAGE_TR", "CPS_AQ"
    ]
    ratio_codes_str = ",".join(ratio_codes)

    url = "https://api-finfo.vndirect.com.vn/v4/ratios"
    params = {
        "q": f"code:{stock_code}~ratioCode:{ratio_codes_str}~reportDate:{report_date}"
    }
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "application/json",
    }

    try:
        response = requests.get(url, params=params, headers=headers, timeout=30)
        response.raise_for_status()
        data = response.json().get("data", [])
        if not data:
            print(f"No financial ratios data found for {stock_code} on {report_date}.")
            return pd.DataFrame()

        df = pd.DataFrame(data)
        if "reportDate" in df.columns:
            df["reportDate"] = pd.to_datetime(df["reportDate"])
        return df
    except requests.exceptions.RequestException as e:
        print(f"Error fetching financial ratios for {stock_code} on {report_date}: {e}")
        return pd.DataFrame()


 



In [7]:
stock_code = "VNM"

# Test fetch_financial_statements
df_statements = fetch_financial_statements(stock_code=stock_code, report_type="QUARTER")
if not df_statements.empty:
    print("Financial Statements (first 5 rows):")
    print(df_statements.head())

Financial Statements (first 5 rows):
  code  itemCode reportType  modelType  numericValue fiscalDate  \
0  VNM   12700.0    QUARTER        1.0  5.331237e+13 2025-12-31   
1  VNM   13000.0    QUARTER        1.0  1.882936e+13 2025-12-31   
2  VNM   13110.0    QUARTER        1.0  9.393737e+12 2025-12-31   
3  VNM   11540.0    QUARTER        1.0  3.008042e+10 2025-12-31   
4  VNM   14330.0    QUARTER        1.0  0.000000e+00 2025-12-31   

           createdDate         modifiedDate  
0  2026-01-30 00:00:00  2026-03-02 00:00:00  
1  2026-01-30 00:00:00  2026-03-02 00:00:00  
2  2026-01-30 00:00:00  2026-03-02 00:00:00  
3  2026-01-30 00:00:00  2026-03-02 00:00:00  
4  2026-01-30 00:00:00  2026-03-02 00:00:00  


In [8]:
stock_code = "VNM"

# Test fetch_financial_models
df_models = fetch_financial_models(stock_code=stock_code)
if not df_models.empty:
    print("Financial Models (first 5 rows):")
    print(df_models.head())

Financial Models (first 5 rows):
   modelType modelTypeName                   modelVnDesc modelEnDesc  \
0        1.0  BALANCESHEET  Bảng Cân đối kế toán - Chung               
1        1.0  BALANCESHEET  Bảng Cân đối kế toán - Chung               
2        1.0  BALANCESHEET  Bảng Cân đối kế toán - Chung               
3        1.0  BALANCESHEET  Bảng Cân đối kế toán - Chung               
4        1.0  BALANCESHEET  Bảng Cân đối kế toán - Chung               

   companyForm               note  \
0  NON_FINANCE  TT202/2014/TT-BTC   
1  NON_FINANCE  TT202/2014/TT-BTC   
2  NON_FINANCE  TT202/2014/TT-BTC   
3  NON_FINANCE  TT202/2014/TT-BTC   
4  NON_FINANCE  TT202/2014/TT-BTC   

                                            codeList  itemCode  \
0  VIN,TT6,VDG,CCC,AVG,AAH,KTW,BCO,NCG,GDA,VLS,BH...   11000.0   
1  VIN,TT6,VDG,CCC,AVG,AAH,KTW,BCO,NCG,GDA,VLS,BH...   11100.0   
2  VIN,TT6,VDG,CCC,AVG,AAH,KTW,BCO,NCG,GDA,VLS,BH...   11110.0   
3  VIN,TT6,VDG,CCC,AVG,AAH,KTW,BCO,NCG,GDA,VLS,

In [9]:
stock_code = "VNM"
start_date = "2026-02-01"
end_date = "2026-03-31"

# Test fetch_stock_price
df_prices = fetch_stock_price(stock_code=stock_code, start_date=start_date, end_date=end_date)
if not df_prices.empty:
    print("Stock Prices (first 5 rows):")
    print(df_prices.head())

Stock Prices (first 5 rows):
  code       date      time floor   type  basicPrice  ceilingPrice  \
0  VNM 2026-03-31  15:03:03  HOSE  STOCK        60.6          64.8   
1  VNM 2026-03-30  15:03:02  HOSE  STOCK        61.5          65.8   
2  VNM 2026-03-27  15:03:03  HOSE  STOCK        61.0          65.2   
3  VNM 2026-03-26  15:03:04  HOSE  STOCK        62.1          66.4   
4  VNM 2026-03-25  15:03:03  HOSE  STOCK        61.2          65.4   

   floorPrice  open  high  ...  adLow  adClose  adAverage   nmVolume  \
0        56.4  61.0  61.5  ...   60.5     60.5     60.874  5323200.0   
1        57.2  60.6  61.5  ...   60.2     60.6     60.738  4075600.0   
2        56.8  61.0  61.9  ...   60.9     61.5     61.344  2137400.0   
3        57.8  62.1  62.1  ...   61.0     61.0     61.365  2044000.0   
4        57.0  61.5  62.2  ...   61.4     62.1     61.903  3445600.0   

        nmValue  ptVolume    ptValue  change  adChange  pctChange  
0  3.240437e+11       0.0        0.0    -0.1     

In [12]:
stock_code = "VNM"
report_date = "2022-12-31"

# Test get_financial_ratios
df_ratios = get_financial_ratios(stock_code=stock_code, report_date=report_date)
if not df_ratios.empty:
    print("Financial Ratios (first 5 rows):")
    print(df_ratios.head())

Financial Ratios (first 5 rows):
  code  group reportDate itemCode                ratioCode  \
0  VNM  STOCK 2022-12-31    53058  OPERATING_EBIT_TR_GRYOY   
1  VNM  STOCK 2022-12-31    65006     INTEREST_COVERAGE_TR   
2  VNM  STOCK 2022-12-31    52005      NET_PROFIT_TR_GRYOY   
3  VNM  STOCK 2022-12-31    53060          CFO_TO_SALES_TR   
4  VNM  STOCK 2022-12-31    56008                   CPS_AQ   

                                            itemName        value  
0  Tăng trưởng EBIT hoạt động 4 quý liền kề so vớ...    -0.170671  
1          Khả năng thanh toán lãi vay 4 quý liền kề    58.740225  
2  Tăng trưởng LN ròng 4 quý liền kề so với cùng ...    -0.191451  
3          Tỷ lệ CFO / doanh thu thuần 4 quý liền kề     0.147229  
4                          Tiền mặt trên 1 cổ phiếu   7072.323158  


In [14]:
stock_code = "VNM"
report_date = "2022-12-31"

# Test get_financial_ratios
df_ratios = get_financial_ratios(stock_code=stock_code, report_date=report_date)
if not df_ratios.empty:
    print("Financial Ratios (first 5 rows):")
    print(df_ratios.head())

Financial Ratios (first 5 rows):
  code  group reportDate itemCode                ratioCode  \
0  VNM  STOCK 2022-12-31    53058  OPERATING_EBIT_TR_GRYOY   
1  VNM  STOCK 2022-12-31    65006     INTEREST_COVERAGE_TR   
2  VNM  STOCK 2022-12-31    52005      NET_PROFIT_TR_GRYOY   
3  VNM  STOCK 2022-12-31    53060          CFO_TO_SALES_TR   
4  VNM  STOCK 2022-12-31    56008                   CPS_AQ   

                                            itemName        value  
0  Tăng trưởng EBIT hoạt động 4 quý liền kề so vớ...    -0.170671  
1          Khả năng thanh toán lãi vay 4 quý liền kề    58.740225  
2  Tăng trưởng LN ròng 4 quý liền kề so với cùng ...    -0.191451  
3          Tỷ lệ CFO / doanh thu thuần 4 quý liền kề     0.147229  
4                          Tiền mặt trên 1 cổ phiếu   7072.323158  
